# TraceletCodeAgent on GAIA

One parameterised notebook covering every Tracelet sweep. Set the config in the next cell and run top to
bottom; output paths are derived from the config, so a config that was already run is served from its
existing `pkl` cache instead of re-running.

To re-run a config that already has cached results, set `RUN_TAG` (e.g. `"_rerun"`) for fresh paths.

In [28]:
# Pick up edits to the editable-installed smolagents without a kernel restart.
%load_ext autoreload
%autoreload 2

# ===================== experiment configuration =====================
MODEL = "gpt-5.4-mini"                 # key of MODELS below
SKELETON_STRATEGY = "direct_prompt"     # "direct_prompt" | "post_process"
N_SAMPLES = 1                          # fill-in candidates per step; 1 = the ablation
USE_TRACELET_PROMPT = True             # tracelet_agent.yaml (template/fill-in protocol) vs default code_agent.yaml
N_QUESTIONS = 10                     # GAIA validation questions; None = all 165
MAX_STEPS = 50
RUN_TAG = "_reasoningprobe"        # e.g. "_rerun" for fresh output paths
# ====================================================================

MODELS = {
    "gpt-4o": "openai",
    "gpt-5.4-mini": "openai",
    "Qwen/Qwen3.5-9B": "together",
    "Qwen/Qwen3.7-Plus": "together",
}
ENABLE_THINKING = False  # Together-served open-weight models only

assert MODEL in MODELS, f"unknown MODEL {MODEL!r}; pick one of {list(MODELS)}"
assert SKELETON_STRATEGY in ("direct_prompt", "post_process")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = MODEL
provider = MODELS[MODEL]

# Only Qwen3.7-Plus rejects non-streaming, and streaming drops fill-in logprobs, so stream only when forced.
STREAM_ONLY = {"Qwen/Qwen3.7-Plus"}

if provider == "openai":
    model = OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
else:
    # Together needs enable_thinking nested in chat_template_kwargs for vLLM-served models (see naiveReAct).
    model = OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
        client_kwargs={"timeout": 300.0},  # bound a stalled stream instead of hanging forever
    )
stream_outputs = model_name in STREAM_ONLY

print(f"{model_name} via {provider} | stream_outputs={stream_outputs}")

In [30]:
# Standard GAIA tool stack, same as naiveReAct.ipynb.
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

In [31]:
import importlib.resources

import yaml

from smolagents.monitoring import LogLevel
from smolagents.tracelet_agent import TraceletCodeAgent

# None lets CodeAgent fall back to its default code_agent.yaml.
prompt_templates = None
if USE_TRACELET_PROMPT:
    prompt_templates = yaml.safe_load(
        importlib.resources.files("smolagents.prompts").joinpath("tracelet_agent.yaml").read_text()
    )

agent = TraceletCodeAgent(
    tools=tools,
    model=model,
    max_steps=MAX_STEPS,
    verbosity_level=LogLevel.DEBUG,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    n_samples=N_SAMPLES,
    skeleton_strategy=SKELETON_STRATEGY,
    stream_outputs=stream_outputs,
    prompt_templates=prompt_templates,
)

In [32]:
# Load GAIA validation set from HuggingFace
import pandas as pd
from common_setup import load_gaia_dataset

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 165 examples


In [ ]:
# Reasoning-token probe: record completion_tokens_details for every OpenAI API call.
import json
from openai.resources.chat.completions import Completions

PROBE_LOG = "/private/tmp/claude-502/-Users-poorvag-Work-smolagents/2a2cc3b2-ba33-45f3-9f0d-3e21f682d237/scratchpad/reasoning_probe.jsonl"
open(PROBE_LOG, "w").close()
_orig_create = Completions.create
_n = {"i": 0}


def _tag(messages):
    last = messages[-1] if messages else {}
    c = last.get("content") if isinstance(last, dict) else None
    text = c if isinstance(c, str) else json.dumps(c)
    if "<sentinel>: <value>" in text:
        return "fillin"
    if "Candidate <i>: <score>" in text:
        return "judge"
    return "skeleton"


def _probe_create(self, *args, **kwargs):
    resp = _orig_create(self, *args, **kwargs)
    u = getattr(resp, "usage", None)
    if u is not None:
        d = getattr(u, "completion_tokens_details", None)
        _n["i"] += 1
        rec = {
            "call": _n["i"],
            "kind": _tag(kwargs.get("messages") or []),
            "prompt_tokens": u.prompt_tokens,
            "completion_tokens": u.completion_tokens,
            "reasoning_tokens": getattr(d, "reasoning_tokens", None),
            "effort_sent": kwargs.get("reasoning_effort", "<omitted>"),
            "model": kwargs.get("model"),
        }
        with open(PROBE_LOG, "a") as f:
            f.write(json.dumps(rec) + "\n")
    return resp


Completions.create = _probe_create
print("reasoning probe armed ->", PROBE_LOG)


In [33]:
from common_setup import evaluate_agent, question_scorer

STRATEGY_TAG = {"direct_prompt": "direct", "post_process": "postprocess"}[SKELETON_STRATEGY]
N_TAG = "" if N_SAMPLES == 3 else f"_n{N_SAMPLES}"
PROMPT_TAG = "_tp" if USE_TRACELET_PROMPT else ""
RUN_NAME = f"tracelet_{STRATEGY_TAG}{N_TAG}{PROMPT_TAG}{RUN_TAG}_react_{model_name}"

# The first sweep predates this naming scheme; keep it pointed at its original directory.
LEGACY_RUN_NAMES = {("gpt-4o", "post_process", 3): "tracelet_react_gpt-4o"}
if not RUN_TAG and not PROMPT_TAG:
    RUN_NAME = LEGACY_RUN_NAMES.get((MODEL, SKELETON_STRATEGY, N_SAMPLES), RUN_NAME)

print(f"run: {RUN_NAME}  (cached examples are reused; set RUN_TAG for a fresh run)")

results = evaluate_agent(
    agent,
    eval_ds,
    ti_tool,
    visualizer,
    n_samples=N_QUESTIONS,
    output_file=f"{RUN_NAME}.jsonl",
    pickle_dir=RUN_NAME,
)

run: tracelet_direct_n1_tp_try3_react_gpt-5.4-mini  (cached examples are reused; set RUN_TAG for a fresh run)

[1/5] A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure w...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three   │
│ axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society │
│ in a Physics and Society article submitted to arXiv.org on August 11, 2016?                                     │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2 memory messages


Thought: ──────────────────────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to identify the AI regulation paper, inspect its figure labels, then determine which label word    
also appears in a Physics and Society arXiv article from 2016. I’ll search the web for the June 2022 arXiv paper   
and the August 11, 2016 Physics and Society article, then compare the axis-end words.                              

...

[Tracelet] no tool-call arguments to sample this step -- direct execution (no fill-in sampling, no judge).

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  distance_km = 363396                                                                                             
  marathon_km = 42.195                                                                                             
  record_seconds = 2 * 3600 + 1 * 60 + 9                                                                           
  pace_seconds_per_km = record_seconds / marathon_km                                                               
  total_seconds = distance_km * pace_seconds_per_km                                                                
  total_hours = total_seconds / 3600                                                                               
  rounded_hours = round(total_hours, -3)                                                                           
  final_answer(rounded_hours)                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 17000.0

[Step 6: Duration 1.92 seconds| Input tokens: 112,502 | Output tokens: 983]

  ✗ | 32.6s | 6 steps | tokens: TokenUsage(input_tokens=112502, output_tokens=983, total_tokens=113485)
    Expected: 17
    Got:      17000.0


In [7]:
df = pd.DataFrame(results)
total = len(df)
correct = df["is_correct"].sum()

print(f"=== TraceletCodeAgent ({SKELETON_STRATEGY}, n_samples={N_SAMPLES}, {model_name}) ===")
print(f"Overall accuracy:   {correct}/{total} = {correct / total:.1%}")
print(f"Avg time per question: {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0)).mean()
print(f"Avg total tokens per question: {total_tokens:,.0f}")

print("\nTool usage (total calls across all questions):")
tool_usage_df = pd.DataFrame(df["tool_usage"].tolist()).sum().sort_values(ascending=False)
for tool, count in tool_usage_df.items():
    if count > 0:
        print(f"  {tool}: {int(count)}")

=== TraceletCodeAgent (post_process, n_samples=1, gpt-4o) ===
Overall accuracy:   13/50 = 26.0%
Avg time per question: 69.9s
Avg steps per question: 12.7
Avg total tokens per question: 215,349

Tool usage (total calls across all questions):
  web_search: 155
  visit_page: 84
  page_down: 81
  find_on_page_ctrl_f: 68
  final_answer: 46
  find_archived_url: 10
  inspect_file_as_text: 10
  find_next: 3
  visualizer: 2
